#  Alternative Credit Scoring Framework for the Creative Economy
### Data Science & Business Intelligence Portfolio Proof-of-Work

---

## Project Overview & Rationale
Traditional banking credit scoring models rely heavily on formal employment history, stable monthly payslips, and tangible asset collateral. Consequently, millions of highly profitable digital content creators, designers, and multimedia entrepreneurs in the modern youth economy are locked out of formal financing.

This project designs and deploys an **Alternative Credit Scoring Baseline** tailored to the creative sector. By evaluating non-traditional, public digital footprints—such as production consistency, audience engagement growth, and community sentiment dynamics—we build a data-driven pipeline that enables financial institutions to confidently extend micro-loans and asset financing while keeping Non-Performing Loan (NPL) risks strictly managed.

---

##  Project Objectives
1. **Data Engineering (Simulation):** Develop a robust, rule-based algorithmic dataset reflecting the core operational performance indicators of 251 digital micro-enterprises.
2. **Predictive Modeling:** Train a machine learning classification engine to accurately map alternative creative metrics to risk probabilities and predict loan default statuses.
3. **Risk Segmentation:** Categorize applicant streams into high, medium, and low-risk strategic tiers to automate early-stage credit approval workflows.
4. **Operational Business Intelligence:** Map these predictive features into an interactive executive dashboard layout to bridge technical data pipelines with business decision-making.

---

##  Tech Stack & Architecture
* **Language:** Python 3
* **Data Manipulation:** Pandas & NumPy
* **Machine Learning Infrastructure:** Scikit-Learn (Decision Tree Classifier Architecture)
* **Visualization Engine:** Power BI / Markdown Matrix Mapping



In [1]:

import random
import pandas as pd

# Fix the random seed so that our generated data remains exactly the same every time we run it
random.seed(42)

# Create an empty list to store each creator's data row during our loop
creatives_list = []

In [2]:

#  LOOP AND GENERATE INDIVIDUAL CREATOR PROFILES

# We loop from 1 to 251 to create exactly 251 unique creative entrepreneurs
for i in range(1, 252):
    # Create a structured, professional tracking ID for each applicant
    creator_id = f"CR-2026-{i:03d}"
    
    # Randomly assign a creative niche sector to the applicant
    niches = ['Digital Video', 'Fashion/Design', 'Music/Audio', 'Fine Art/Photography']
    niche = random.choice(niches)
    
    # Simulate non-traditional digital footprints as percentages/decimals
    upload_consistency = round(random.uniform(0.30, 0.99), 2)  # Channel upload consistency index
    view_growth_rate = round(random.uniform(-0.15, 0.60), 2)    # 90-day audience traffic growth rate
    audience_sentiment = round(random.uniform(0.20, 0.95), 2)   # Ratio of positive comments to negative comments
    
    # Assign a requested micro-loan amount in Kenyan Shillings (KES)
    loan_amounts = [50000, 100000, 150000, 250000, 500000]
    requested_loan = random.choice(loan_amounts)
    
    # Calculate a custom performance risk score based on poor metrics
    # Lower consistency and negative growth rates yield a higher risk score
    poor_performance_score = (1 - upload_consistency) + (0.60 - view_growth_rate)
    
    # Apply business logic rules to determine the ground-truth repayment status
    if poor_performance_score > 1.0:
        loan_default_status = 1  # High Risk: Model predicts potential default
    elif poor_performance_score < 0.4:
        loan_default_status = 0  # Low Risk: Model predicts successful repayment
    else:
        # In the middle ground, create a realistic variance using a 50/50 coin flip
        loan_default_status = random.choice([0, 1])
        
    # Bundle all individual variables into a single dictionary profile
    creator_row = {
        'creator_id': creator_id,
        'creative_niche': niche,
        'upload_consistency_index': upload_consistency,
        'view_growth_rate_90d': view_growth_rate,
        'audience_sentiment_score': audience_sentiment,
        'requested_loan_amount_kes': requested_loan,
        'loan_default_status': loan_default_status
    }
    
    # Append the dictionary row to our master list
    creatives_list.append(creator_row)

In [3]:

#  CONVERT TO DATAFRAME AND VALIDATE SHAPE

# Convert our list of compiled dictionaries into a structured data table
df = pd.DataFrame(creatives_list)

# Save this raw dataset safely onto our machine as a CSV file
df.to_csv('creative_loan_data_raw.csv', index=False)


In [4]:
df.shape

(251, 7)

## Data Preprocessing and Machine Learning Model Training

**Objective:** In this phase, we read our generated dataset back into memory, transform our categorical text features into numerical formats using One-Hot Encoding, partition our data to prevent bias, and train a Decision Tree Classifier to evaluate loan default risks.

In [6]:

# LOAD THE  DATASET
df = pd.read_csv('creative_loan_data_raw.csv')
df

,creator_id,creative_niche,upload_consistency_index,view_growth_rate_90d,audience_sentiment_score,requested_loan_amount_kes,loan_default_status
0,CR-2026-001,Digital Video,0.32,0.06,0.37,50000,1
1,CR-2026-002,Digital Video,0.71,-0.13,0.27,100000,1
2,CR-2026-003,Digital Video,0.69,0.39,0.73,250000,0
3,CR-2026-004,Fine Art/Photography,0.71,0.46,0.20,100000,1
4,CR-2026-005,Music/Audio,0.49,0.01,0.77,50000,1
...,...,...,...,...,...,...,...
246,CR-2026-247,Fine Art/Photography,0.36,0.53,0.81,150000,0
247,CR-2026-248,Fine Art/Photography,0.34,0.23,0.38,50000,1
248,CR-2026-249,Music/Audio,0.45,0.09,0.27,150000,1
249,CR-2026-250,Fashion/Design,0.39,0.07,0.82,100000,1


In [10]:
df.columns

Index(['creator_id', 'creative_niche', 'upload_consistency_index',
       'view_growth_rate_90d', 'audience_sentiment_score',
       'requested_loan_amount_kes', 'loan_default_status'],
      dtype='object')

###  Feature Engineering (One-Hot Encoding)
Machine learning models are mathematical calculators that do not natively understand string metrics like "Digital Video" or "Music/Audio". 

To handle this, we apply **One-Hot Encoding**. This technique expands our text categories into distinct binary columns (containing `1` for True or `0` for False). This allows our model to learn patterns across different creative niches without assigning an arbitrary mathematical weight or ranking to the names themselves.

In [11]:

# APPLY ONE-HOT ENCODING
# Convert the 'creative_niche' text column into multiple numerical binary columns
df_encoded = pd.get_dummies(df, columns=['creative_niche'])
print(list(df_encoded.columns))

['creator_id', 'upload_consistency_index', 'view_growth_rate_90d', 'audience_sentiment_score', 'requested_loan_amount_kes', 'loan_default_status', 'creative_niche_Digital Video', 'creative_niche_Fashion/Design', 'creative_niche_Fine Art/Photography', 'creative_niche_Music/Audio']


###  Dataset Splitting & Model Training
To ensure our model genuinely learns underlying patterns instead of just memorizing our dataset (a common pitfall known as **overfitting**), we split our data using an **80/20 train-test split**:
* **80% Training Set:** The historical data the model uses to discover rules and calculate credit risk logic.
* **20% Testing Set:** Completely unseen validation data used to evaluate how accurately the model handles new applicants.

We use a **Decision Tree Classifier**, which constructs a logical flowchart based on variables like consistency indices and growth rates to output a binary decision (`0` for repayment, `1` for default).

In [13]:

# TESTING AND TRAINING THE MODEL

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = df_encoded.drop(columns=['creator_id', 'loan_default_status'])
y = df_encoded['loan_default_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train, y_train)


predictions = model.predict(X_test)
testing_accuracy = accuracy_score(y_test, predictions)

print(f"📊 Model Accuracy Baseline on Unseen Test Data: {testing_accuracy * 100:.1f}%")

📊 Model Accuracy Baseline on Unseen Test Data: 64.7%


### Scoring the Full Dataset & Risk Segmentation
To make our machine learning model useful for Equity Bank's credit officers, we will apply our trained model back to our entire dataset. 

Instead of a simple binary prediction, we will generate a **Risk Probability Score (0.0 to 1.0)** for every applicant. We will then use Pandas to segment these scores into three actionable business tiers:
* **Low Risk:** Automatically approved for review.
* **Medium Risk:** Sent to a credit officer for manual verification.
* **High Risk:** Automatically flagged for rejection.

In [14]:

# GENERATE RISK TIERS & EXPORT DATA
import numpy as np
# 1. Generate risk probabilities for the entire dataset
# predict_proba returns two columns: [prob of 0, prob of 1]. We extract column 1 (prob of default).
df['default_probability'] = np.round(model.predict_proba(X)[:, 1], 3)

# 2. Segment creators into business tiers based on their probability score
df['credit_risk_tier'] = pd.cut(
    df['default_probability'],
    bins=[-0.001, 0.40, 0.70, 1.00],
    labels=['Low Risk (Approve)', 'Medium Risk (Review)', 'High Risk (Reject)']
)

# 3. Save this final scored dataset as a new CSV for Power BI
df.to_csv('final_scored_creatives.csv', index=False)

print(df[['creator_id', 'creative_niche', 'default_probability', 'credit_risk_tier']].head())

    creator_id        creative_niche  default_probability  \
0  CR-2026-001         Digital Video                1.000   
1  CR-2026-002         Digital Video                1.000   
2  CR-2026-003         Digital Video                0.429   
3  CR-2026-004  Fine Art/Photography                1.000   
4  CR-2026-005           Music/Audio                1.000   

       credit_risk_tier  
0    High Risk (Reject)  
1    High Risk (Reject)  
2  Medium Risk (Review)  
3    High Risk (Reject)  
4    High Risk (Reject)  


## Summary & Model Conclusion

###  Performance Evaluation
Our baseline **Decision Tree Classifier** achieved a testing accuracy of **64.7%** on unseen validation data. 

In a traditional tech project, a higher accuracy is always preferred, but in a financial risk setting, a 64.7% baseline for an alternative data model indicates a strong foundation. It proves that non-traditional digital footprints—specifically **production consistency** and **audience traffic growth**—contain statistically significant predictive patterns for credit risk. 

###  Banking & Operational Impact
Instead of outputting a rigid "Yes/No" decision, we successfully converted our model's internal classification logic into a continuous **Default Probability Score (0% to 100%)**. 

By segmenting these scores into actionable business tiers (`Low Risk`, `Medium Risk`, and `High Risk`), we create a framework that can transform Equity's operational workflow:
1. **Automation:** `Low Risk` applicants can be Fast-Tracked for instant mobile disbursement.
2. **Efficiency:** `Medium Risk` applicants are routed directly to credit officers for manual verification, reducing their administrative backlog.
3. **Protection:** `High Risk` profiles are flagged early, actively protecting the bank's asset quality and mitigating Non-Performing Loans (NPLs).

